# 02 — Localizar problemas para a EJA

Objetivo: adaptar uma amostra do GSM8K para situações brasileiras, sem mudar os números nem a resposta. Entrada: `data/raw/gsm8k_extraido.jsonl`. Saída: `data/interim/problemas_localizados.jsonl`. Em `MOCK`, os registros são apenas testes técnicos e não exemplos pedagógicos para treino. Em `REAL`, usa Groq via API compatível com OpenAI; a função `chamar_llm` é o único ponto de integração.

In [1]:
# Opcional: %pip install openai python-dotenv "pydantic>=2.7"
import json
import os
import random
import re
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict

def localizar_projeto() -> Path:
    '''Encontra a pasta de dados a partir do diretório corrente.'''
    for base in (Path.cwd(), *Path.cwd().parents):
        pasta = base / 'notebooks' / 'eja_diagnosticador_dataset'
        if (pasta / 'data' / 'config.json').exists():
            return pasta
        if base.name == 'eja_diagnosticador_dataset' and (base / 'data' / 'config.json').exists():
            return base
    raise FileNotFoundError('Configuração compartilhada não encontrada')

ROOT = localizar_projeto()
CONFIG = json.loads((ROOT / 'data' / 'config.json').read_text(encoding='utf-8'))
load_dotenv(ROOT.parent.parent / '.env')
random.seed(CONFIG['seed'])
ENTRADA = ROOT / 'data' / 'raw' / 'gsm8k_extraido.jsonl'
SAIDA = ROOT / 'data' / 'interim' / 'problemas_localizados.jsonl'
CONTEXTOS = ['feira e compras', 'obra e medidas', 'salário e trabalho', 'ônibus e transporte', 'receita e cozinha']
NUMERO = re.compile(r'(?<![A-Za-z])[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?|(?<![A-Za-z])[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?|(?<![A-Za-z])[-+]?\d+(?:[,.]\d+)?')

class Localizacao(BaseModel):
    '''Campos obrigatórios retornados pelo adaptador de linguagem.'''
    model_config = ConfigDict(extra='forbid')
    problema_localizado: str
    contexto_cotidiano: str
    tema: str

class RegistroLocalizado(BaseModel):
    '''Valida campos essenciais e preserva os campos de origem GSM8K.'''
    model_config = ConfigDict(extra='allow')
    id: str
    base_id: str
    problema_localizado: str
    contexto_cotidiano: str
    resposta_numerica: str
    modo_geracao: str

def serializar_jsonl(registro: dict[str, Any]) -> str:
    '''Preserva JSONL de uma linha mesmo com separadores Unicode no texto.'''
    texto = json.dumps(registro, ensure_ascii=False)
    for caractere, escape in ((chr(0x85), r'\u0085'), (chr(0x2028), r'\u2028'), (chr(0x2029), r'\u2029')):
        texto = texto.replace(caractere, escape)
    return texto

def numero_decimal(texto: str) -> Decimal:
    '''Lê número inteiro ou decimal simples sem executar código.'''
    limpo = texto.strip()
    if re.fullmatch(r'[-+]?\d{1,3}(,\d{3})+(\.\d+)?', limpo):
        limpo = limpo.replace(',', '')
    elif re.fullmatch(r'[-+]?\d{1,3}(\.\d{3})+(,\d+)?', limpo):
        limpo = limpo.replace('.', '').replace(',', '.')
    else:
        limpo = limpo.replace(',', '.')
    try:
        return Decimal(limpo)
    except InvalidOperation as erro:
        raise ValueError(f'Número inválido: {texto}') from erro

def numeros(texto: str) -> Counter[Decimal]:
    '''Conta os valores numéricos literais de um texto.'''
    return Counter(numero_decimal(m.group()) for m in NUMERO.finditer(texto))

def resposta_numerica(texto: str) -> Decimal:
    '''Extrai o último valor numérico após o marcador final do GSM8K.'''
    achados = [m.group() for m in NUMERO.finditer(texto)]
    if not achados:
        raise ValueError('Resposta final sem número reconhecível')
    return numero_decimal(achados[-1])

def chamar_llm(pergunta: str, contexto: str) -> dict[str, str]:
    '''Adapta o enunciado; não calcula nem decide respostas matemáticas.'''
    if CONFIG['llm_mode'].upper() == 'MOCK':
        return Localizacao.model_validate({'problema_localizado': f'Exemplo técnico MOCK (não usar para treino). Texto de origem: {pergunta}',
                'contexto_cotidiano': contexto, 'tema': 'teste de pipeline'}).model_dump()
    if CONFIG['llm_mode'].upper() != 'REAL':
        raise ValueError('llm_mode deve ser MOCK ou REAL')
    from openai import OpenAI
    api_key = os.getenv('GROQ_API_KEY')
    if not api_key:
        raise RuntimeError('Defina GROQ_API_KEY no ambiente ou no .env da raiz do projeto')
    cliente = OpenAI(base_url=CONFIG['llm_base_url'], api_key=api_key)
    instrucao = ('Adapte para português brasileiro simples e respeitoso para adultos da EJA. '
        'Use situação cotidiana brasileira adequada e nomes fictícios, frases curtas. '
        'Não infantilize. Preserve exatamente os números e operações do enunciado. '
        'Não inclua nem calcule a resposta. Retorne somente JSON com problema_localizado, '
        'contexto_cotidiano e tema.')
    retorno = cliente.chat.completions.create(model=CONFIG['llm_model'], temperature=0,
        response_format={'type': 'json_object'},
        messages=[{'role': 'system', 'content': instrucao}, {'role': 'user', 'content': pergunta}])
    conteudo = retorno.choices[0].message.content
    if not conteudo:
        raise ValueError('O LLM retornou conteúdo vazio')
    resultado = json.loads(conteudo)
    for campo in ('problema_localizado', 'contexto_cotidiano', 'tema'):
        if not isinstance(resultado.get(campo), str) or not resultado[campo].strip():
            raise ValueError(f'Campo inválido na saída do LLM: {campo}')
    return Localizacao.model_validate(resultado).model_dump()

if not ENTRADA.exists():
    raise FileNotFoundError(f'Execute primeiro o notebook 01: {ENTRADA}')
originais = [json.loads(l) for l in ENTRADA.read_text(encoding='utf-8').splitlines() if l]
amostra = originais[:int(CONFIG['max_examples'])]
SAIDA.parent.mkdir(parents=True, exist_ok=True)
gerados = descartados = 0
with SAIDA.open('w', encoding='utf-8') as arquivo:
    for origem in amostra:
        contexto = random.choice(CONTEXTOS)
        try:
            localizado = chamar_llm(origem['pergunta_original'], contexto)
            # A resposta e todos os números do enunciado precisam permanecer iguais.
            resposta_calculada = resposta_numerica(origem['resposta_final'])
            if numeros(origem['pergunta_original']) - numeros(localizado['problema_localizado']):
                raise ValueError('A localização omitiu ou alterou números do enunciado')
            registro = {**origem, **localizado, 'contexto_cotidiano': localizado.get('contexto_cotidiano', contexto),
                        'resposta_numerica': str(resposta_calculada),
                        'modo_geracao': CONFIG['llm_mode'].upper()}
            registro = RegistroLocalizado.model_validate(registro).model_dump()
            arquivo.write(serializar_jsonl(registro) + '\n')
            gerados += 1
        except (ValueError, KeyError, json.JSONDecodeError) as erro:
            descartados += 1
            print(f"Descartado {origem.get('id')}: {erro}")
print(f'Amostra: {len(amostra)} | gerados: {gerados} | descartados: {descartados} | modo: {CONFIG["llm_mode"]}')

Amostra: 100 | gerados: 100 | descartados: 0 | modo: MOCK


A localização pode mudar o cenário e a redação, mas não os valores matemáticos. Como os números são preservados ou o exemplo é descartado, a calculadora não precisa aceitar uma resposta recalculada pelo modelo. A verificação a seguir também confere a resposta numérica de referência.

In [2]:
localizados = [json.loads(l) for l in SAIDA.read_text(encoding='utf-8').splitlines() if l]
assert len(localizados) == gerados
assert all(r['modo_geracao'] in ('MOCK', 'REAL') and r['problema_localizado'] for r in localizados)
print('Registros:', len(localizados), '| modo:', CONFIG['llm_mode'], '| descartados:', descartados)
if localizados:
    amostra_verificacao = localizados[0]
    assert Decimal(amostra_verificacao['resposta_numerica']) == resposta_numerica(amostra_verificacao['resposta_final'])
    print(json.dumps({k: amostra_verificacao[k] for k in ('id', 'problema_localizado', 'contexto_cotidiano', 'resposta_numerica', 'modo_geracao')}, ensure_ascii=False, indent=2))

Registros: 100 | modo: MOCK | descartados: 0
{
  "id": "train:0",
  "problema_localizado": "Exemplo técnico MOCK (não usar para treino). Texto de origem: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
  "contexto_cotidiano": "feira e compras",
  "resposta_numerica": "72",
  "modo_geracao": "MOCK"
}
